# Fiche de révision Machine Learning — TP + Annales

Cette fiche est faite pour répondre rapidement aux questions d’examen à partir des méthodes vues dans les TP :  
régression, classification, réseaux de neurones, softmax, one-hot encoding, régularisation, séparation des datasets, validation croisée, K-means et PCA.

---

## Méthode générale devant une question

1. Identifier le type de problème : régression, classification binaire, classification multiclasses.
2. Identifier les dimensions : nombre de variables d’entrée, nombre de données, nombre de classes.
3. Choisir la bonne sortie :
   - régression : sortie linéaire ;
   - binaire : sigmoïde ;
   - multiclasses : softmax.
4. Choisir la bonne fonction coût :
   - régression : erreur quadratique moyenne ;
   - classification : cross-entropy.
5. Écrire les dimensions de toutes les matrices.
6. Écrire la propagation avant.
7. Écrire la backpropagation.
8. Ajouter la régularisation seulement si un paramètre $\lambda$ apparaît.
9. Séparer train, validation et test selon la question.
10. Donner la métrique finale : MSE ou accuracy.


# 1. Reconnaître le type de problème

| Type de problème | Situation | Sortie du réseau | Fonction coût | Mesure finale |
|---|---|---|---|---|
| Régression | On prédit une valeur réelle | $a^{(3)} = z^{(3)}$ | MSE | MSE |
| Classification binaire scalaire | $y \in \{0,1\}$ | $a^{(3)} = \sigma(z^{(3)})$ | Binary cross-entropy | Accuracy |
| Classification binaire vectorielle | $y = (1,0)^T$ ou $(0,1)^T$ | $a^{(3)} \in \mathbb{R}^2$ | Cross-entropy | Accuracy |
| Classification multiclasses | Plusieurs classes | $a^{(3)} = softmax(z^{(3)})$ | Cross-entropy multiclasses | Accuracy |
| Choix d’hyperparamètres | Plusieurs modèles à tester | Modèle entraîné | Comparaison validation | Test final |
| Régularisation | Présence de $\lambda$ | Même sortie | Coût + terme régularisé | Généralisation |


# 2. Dimensions générales d’un réseau à 3 couches

On suppose que les données sont stockées en colonnes :

$$
X \in \mathbb{R}^{n_1 \times m}
$$

où :

- $n_1$ est le nombre de variables d’entrée ;
- $m$ est le nombre d’observations ;
- chaque colonne de $X$ est une observation.

Le réseau est :

$$
a^{(1)} = X
$$

$$
z^{(2)} = W^{(2)}a^{(1)} + b^{(2)}
$$

$$
a^{(2)} = g(z^{(2)})
$$

$$
z^{(3)} = W^{(3)}a^{(2)} + b^{(3)}
$$

La dernière activation dépend du problème.

| Objet | Dimension |
|---|---|
| $X = a^{(1)}$ | $(n_1, m)$ |
| $W^{(2)}$ | $(n_2, n_1)$ |
| $b^{(2)}$ | $(n_2, 1)$ |
| $z^{(2)}, a^{(2)}$ | $(n_2, m)$ |
| $W^{(3)}$ | $(n_3, n_2)$ |
| $b^{(3)}$ | $(n_3, 1)$ |
| $z^{(3)}, a^{(3)}, Y$ | $(n_3, m)$ |

Vérification des produits matriciels :

$$
W^{(2)}a^{(1)}
\in
\mathbb{R}^{n_2 \times n_1}
\mathbb{R}^{n_1 \times m}
=
\mathbb{R}^{n_2 \times m}
$$

$$
W^{(3)}a^{(2)}
\in
\mathbb{R}^{n_3 \times n_2}
\mathbb{R}^{n_2 \times m}
=
\mathbb{R}^{n_3 \times m}
$$


In [ ]:
def dimension_summary(n1, n2, n3, m):
    """
    Résumé des dimensions pour un réseau à 3 couches.
    n1 : nombre de variables d'entrée
    n2 : nombre de neurones cachés
    n3 : nombre de sorties/classes
    m  : nombre d'observations
    """
    return {
        "X = A1": (n1, m),
        "W2": (n2, n1),
        "b2": (n2, 1),
        "Z2 = A2": (n2, m),
        "W3": (n3, n2),
        "b3": (n3, 1),
        "Z3 = A3 = Y": (n3, m),
    }

dimension_summary(n1=14, n2=20, n3=1, m=1000)


# 3. Régression

On utilise la régression quand la sortie attendue est une valeur réelle.

Exemples :

- prix d’une maison ;
- note d’un étudiant ;
- quantité ;
- valeur numérique continue.

## Modèle

La couche cachée utilise une activation sigmoïde :

$$
a^{(2)} = \sigma(z^{(2)})
$$

Mais la sortie est linéaire :

$$
a^{(3)} = z^{(3)}
$$

Donc il ne faut pas mettre de sigmoïde ou de softmax en sortie.

## Dimensions

Si $X \in \mathbb{R}^{n_1 \times m}$ et si la couche cachée contient $n_2$ neurones :

| Objet | Dimension |
|---|---|
| $X$ | $(n_1, m)$ |
| $Y$ | $(1, m)$ |
| $W^{(2)}$ | $(n_2, n_1)$ |
| $b^{(2)}$ | $(n_2, 1)$ |
| $W^{(3)}$ | $(1, n_2)$ |
| $b^{(3)}$ | $(1, 1)$ |
| $a^{(3)}$ | $(1, m)$ |

## Fonction coût

L’erreur quadratique moyenne est :

$$
J = \frac{1}{m}\sum_{i=1}^{m}
\left(a^{(3)}(x^{(i)}) - y^{(i)}\right)^2
$$

En Python :

```python
J = np.mean((A3 - Y)**2)
```

## Modifications à faire dans un code de classification

Pour transformer un réseau de classification en réseau de régression :

1. mettre une seule sortie ;
2. prendre $W^{(3)} \in \mathbb{R}^{1 \times n_2}$ ;
3. prendre $b^{(3)} \in \mathbb{R}^{1 \times 1}$ ;
4. remplacer la dernière activation par $a^{(3)} = z^{(3)}$ ;
5. remplacer le coût par la MSE ;
6. remplacer l’accuracy par la MSE.


In [ ]:
import numpy as np

def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

def forward_regression(X, W2, b2, W3, b3):
    A1 = X
    Z2 = W2 @ A1 + b2
    A2 = sigmoid(Z2)
    Z3 = W3 @ A2 + b3
    A3 = Z3
    return A1, Z2, A2, Z3, A3

def mse(A3, Y):
    return np.mean((A3 - Y)**2)


# 4. Classification binaire avec sortie scalaire

On utilise ce cas quand :

$$
Y \in \{0,1\}^{1 \times m}
$$

## Modèle

La sortie est une sigmoïde :

$$
a^{(3)} = \sigma(z^{(3)})
$$

avec :

$$
\sigma(z) = \frac{1}{1+e^{-z}}
$$

Comme la sigmoïde renvoie une valeur entre 0 et 1, on peut interpréter $a^{(3)}$ comme une probabilité.

## Décision

La règle de décision est :

$$
\hat{y} =
\begin{cases}
1 & \text{si } a^{(3)} > 0.5 \\
0 & \text{sinon}
\end{cases}
$$

En Python :

```python
Y_pred = (A3 > 0.5)
```

## Fonction coût

La binary cross-entropy est :

$$
J =
-\frac{1}{m}
\sum_{i=1}^{m}
\left[
y^{(i)}\log(a^{(3)}(x^{(i)}))
+
(1-y^{(i)})\log(1-a^{(3)}(x^{(i)}))
\right]
$$

## Résultat important pour la backpropagation

Avec sigmoïde en sortie et binary cross-entropy :

$$
\Delta^{(3)} = a^{(3)} - Y
$$


In [ ]:
def binary_cross_entropy(A3, Y):
    m = Y.shape[1]
    eps = 1e-12
    return -(1/m) * np.sum(Y*np.log(A3 + eps) + (1-Y)*np.log(1-A3 + eps))

def predict_binary(A3):
    return (A3 > 0.5).astype(int)

def accuracy_binary(A3, Y):
    Y_pred = predict_binary(A3)
    return np.mean(Y_pred == Y)


# 5. Classification binaire avec sortie vectorielle

On utilise ce cas quand les deux classes sont codées par des vecteurs.

Exemple :

$$
\text{classe 1}
=
\begin{pmatrix}
1 \\
0
\end{pmatrix}
$$

$$
\text{classe 2}
=
\begin{pmatrix}
0 \\
1
\end{pmatrix}
$$

Alors :

$$
Y \in \mathbb{R}^{2 \times m}
$$

## Dimensions

| Objet | Dimension |
|---|---|
| $X$ | $(n_1, m)$ |
| $Y$ | $(2, m)$ |
| $W^{(2)}$ | $(n_2, n_1)$ |
| $b^{(2)}$ | $(n_2, 1)$ |
| $W^{(3)}$ | $(2, n_2)$ |
| $b^{(3)}$ | $(2, 1)$ |
| $a^{(3)}$ | $(2, m)$ |

## Décision

On choisit la composante la plus grande :

$$
\hat{y} = \arg\max_k a^{(3)}_k
$$

En Python :

```python
prediction = np.argmax(A3, axis=0)
vrai_label = np.argmax(Y, axis=0)
accuracy = np.mean(prediction == vrai_label)
```


In [ ]:
def accuracy_vector_output(A3, Y):
    pred = np.argmax(A3, axis=0)
    true = np.argmax(Y, axis=0)
    return np.mean(pred == true)


# 6. Softmax

Softmax est utilisée pour la classification multiclasses.

Si le réseau a $K$ sorties, alors :

$$
a^{(3)} = softmax(z^{(3)})
$$

Pour chaque classe $k$ :

$$
softmax(z)_k =
\frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}
$$

## Propriété importante

Les composantes de softmax somment à 1 :

$$
\sum_{k=1}^{K} softmax(z)_k = 1
$$

Donc la sortie peut être interprétée comme un vecteur de probabilités.

## Fonction coût softmax + cross-entropy

Si $Y$ est one-hot encodé :

$$
J =
-\frac{1}{m}
\sum_{i=1}^{m}
\sum_{k=1}^{K}
y_k^{(i)}
\log(a_k^{(3)}(x^{(i)}))
$$

## Backpropagation en dernière couche

Avec softmax et cross-entropy :

$$
\Delta^{(3)} = a^{(3)} - Y
$$


In [ ]:
def softmax(Z):
    """
    Softmax stable numériquement.
    Les colonnes représentent les observations.
    """
    expZ = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return expZ / np.sum(expZ, axis=0, keepdims=True)

def cross_entropy_softmax(A3, Y):
    m = Y.shape[1]
    eps = 1e-12
    return -(1/m) * np.sum(Y * np.log(A3 + eps))


# 7. One-hot encoding

Le one-hot encoding transforme un label scalaire en vecteur.

Exemple avec $K=4$ classes :

$$
y = 2
\quad \Longrightarrow \quad
\begin{pmatrix}
0 \\
0 \\
1 \\
0
\end{pmatrix}
$$

Si on a $m$ observations et $K$ classes :

$$
Y \in \mathbb{R}^{K \times m}
$$

## Exemple

Si les labels sont :

$$
y =
\begin{pmatrix}
0 & 2 & 1 & 2
\end{pmatrix}
$$

alors le one-hot encoding donne :

$$
Y =
\begin{pmatrix}
1 & 0 & 0 & 0 \\
0 & 0 & 1 & 0 \\
0 & 1 & 0 & 1
\end{pmatrix}
$$

Chaque colonne correspond à une observation.


In [ ]:
def one_hot(y, K):
    """
    Transforme un vecteur de labels y en matrice one-hot.
    y doit contenir des entiers entre 0 et K-1.
    """
    y = np.asarray(y).astype(int).ravel()
    Y = np.zeros((K, y.size))
    Y[y, np.arange(y.size)] = 1
    return Y

# Exemple
one_hot(np.array([0, 2, 1, 2]), K=3)


# 8. Backpropagation pour un réseau à 3 couches

## Propagation avant

Pour une classification multiclasses :

$$
a^{(1)} = X
$$

$$
z^{(2)} = W^{(2)}a^{(1)} + b^{(2)}
$$

$$
a^{(2)} = \sigma(z^{(2)})
$$

$$
z^{(3)} = W^{(3)}a^{(2)} + b^{(3)}
$$

$$
a^{(3)} = softmax(z^{(3)})
$$

## Backpropagation

En dernière couche :

$$
\Delta^{(3)} = a^{(3)} - Y
$$

Gradient par rapport à $W^{(3)}$ :

$$
\frac{\partial J}{\partial W^{(3)}} =
\frac{1}{m}
\Delta^{(3)}(a^{(2)})^T
$$

Gradient par rapport à $b^{(3)}$ :

$$
\frac{\partial J}{\partial b^{(3)}} =
\frac{1}{m}
\sum_{i=1}^{m}
\Delta_i^{(3)}
$$

Propagation de l’erreur vers la couche cachée :

$$
\Delta^{(2)}
=
(W^{(3)})^T
\Delta^{(3)}
\odot
\sigma'(z^{(2)})
$$

avec :

$$
\sigma'(z) = \sigma(z)(1-\sigma(z))
$$

Gradient par rapport à $W^{(2)}$ :

$$
\frac{\partial J}{\partial W^{(2)}} =
\frac{1}{m}
\Delta^{(2)}(a^{(1)})^T
$$

Gradient par rapport à $b^{(2)}$ :

$$
\frac{\partial J}{\partial b^{(2)}} =
\frac{1}{m}
\sum_{i=1}^{m}
\Delta_i^{(2)}
$$


In [ ]:
def sigmoid_prime_from_activation(A):
    return A * (1 - A)

def forward_softmax(X, W2, b2, W3, b3):
    A1 = X
    Z2 = W2 @ A1 + b2
    A2 = sigmoid(Z2)
    Z3 = W3 @ A2 + b3
    A3 = softmax(Z3)
    return A1, Z2, A2, Z3, A3

def backward_softmax(A1, Z2, A2, A3, Y, W2, W3):
    m = Y.shape[1]

    dZ3 = A3 - Y
    dW3 = (1/m) * dZ3 @ A2.T
    db3 = (1/m) * np.sum(dZ3, axis=1, keepdims=True)

    dZ2 = W3.T @ dZ3 * sigmoid_prime_from_activation(A2)
    dW2 = (1/m) * dZ2 @ A1.T
    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)

    return dW2, db2, dW3, db3

def update_parameters(W2, b2, W3, b3, dW2, db2, dW3, db3, alpha):
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2
    W3 = W3 - alpha * dW3
    b3 = b3 - alpha * db3
    return W2, b2, W3, b3


# 9. Régularisation

On ajoute la régularisation lorsqu’un paramètre $\lambda$ apparaît.

La régularisation s’applique aux poids, mais pas aux biais.

## Coût régularisé

Pour un réseau avec deux matrices de poids :

$$
J_{reg}
=
J
+
\frac{\lambda}{2m}
\left(
\sum_{i,j}(W_{ij}^{(2)})^2
+
\sum_{i,j}(W_{ij}^{(3)})^2
\right)
$$

Pour un réseau avec trois matrices de poids :

$$
J_{reg}
=
J
+
\frac{\lambda}{2m}
\left(
\sum_{i,j}(W_{ij}^{(2)})^2
+
\sum_{i,j}(W_{ij}^{(3)})^2
+
\sum_{i,j}(W_{ij}^{(4)})^2
\right)
$$

## Gradients régularisés

Pour les poids :

$$
\frac{\partial J_{reg}}{\partial W^{(l)}}
=
\frac{\partial J}{\partial W^{(l)}}
+
\frac{\lambda}{m}W^{(l)}
$$

Pour les biais :

$$
\frac{\partial J_{reg}}{\partial b^{(l)}}
=
\frac{\partial J}{\partial b^{(l)}}
$$

Les biais ne sont donc pas régularisés.


In [ ]:
def add_regularization_to_cost(cost, weights, lambda_, m):
    """
    weights : liste de matrices de poids, par exemple [W2, W3] ou [W2, W3, W4].
    """
    reg = sum(np.sum(W**2) for W in weights)
    return cost + (lambda_/(2*m)) * reg

def regularize_gradient(dW, W, lambda_, m):
    return dW + (lambda_/m) * W


# 10. Train, validation et test

## Training set

Le training set sert à apprendre les paramètres du modèle :

$$
W^{(2)}, b^{(2)}, W^{(3)}, b^{(3)}
$$

## Validation set

Le validation set sert à choisir les hyperparamètres :

$$
n_2,\quad n_3,\quad \alpha,\quad \lambda,\quad N_{iter}
$$

## Test set

Le test set sert seulement à la fin pour estimer la performance finale du modèle.

## Réponse type : pourquoi pas besoin de validation set ?

On n’a pas besoin de validation set si les hyperparamètres sont déjà imposés par l’énoncé.  
Le validation set sert à choisir les hyperparamètres.  
Si $n_2$, $\alpha$, $\lambda$ et $N_{iter}$ sont fixés, il n’y a rien à choisir.

## Réponse type : pourquoi mélanger les données ?

On mélange les données avant de séparer train/test parce que les données peuvent être rangées par classe ou par ordre particulier.  
Sans mélange, le train et le test peuvent ne pas être représentatifs.


In [ ]:
from sklearn.utils import shuffle

def split_train_test_columns(X, Y, train_size):
    """
    Séparation de données stockées en colonnes.
    X : shape (n_features, m)
    Y : shape (n_outputs, m)
    train_size : nombre de colonnes pour le train
    """
    X_train = X[:, :train_size]
    Y_train = Y[:, :train_size]
    X_test = X[:, train_size:]
    Y_test = Y[:, train_size:]
    return X_train, Y_train, X_test, Y_test

def minmax_normalize_train_test(X_train, X_test):
    """
    Normalisation entre 0 et 1.
    Les min et max sont calculés uniquement sur le train.
    """
    Xmin = np.min(X_train, axis=1, keepdims=True)
    Xmax = np.max(X_train, axis=1, keepdims=True)
    denom = Xmax - Xmin
    denom[denom == 0] = 1
    X_train_norm = (X_train - Xmin) / denom
    X_test_norm = (X_test - Xmin) / denom
    return X_train_norm, X_test_norm


# 11. Accuracy et MSE

## Accuracy pour classification multiclasses

Si la sortie est softmax :

$$
\hat{y}^{(i)}
=
\arg\max_k a_k^{(3)}(x^{(i)})
$$

L’accuracy est :

$$
accuracy
=
\frac{\text{nombre de prédictions correctes}}{\text{nombre total de prédictions}}
$$

## MSE pour régression

Pour une régression :

$$
MSE =
\frac{1}{m}
\sum_{i=1}^{m}
(\hat{y}^{(i)} - y^{(i)})^2
$$


In [ ]:
def accuracy_multiclass(A, Y):
    pred = np.argmax(A, axis=0)
    true = np.argmax(Y, axis=0)
    return np.mean(pred == true)

def mse_metric(Y_pred, Y_true):
    return np.mean((Y_pred - Y_true)**2)


# 12. Validation croisée en 5 blocs

Si on a $900$ données d’entraînement et que l’on fait une validation croisée en $5$ blocs :

$$
\frac{900}{5} = 180
$$

Chaque modèle utilise :

- $720$ données pour l’entraînement ;
- $180$ données pour le test interne.

Pour chaque bloc $k$ :

| Objet | Dimension en régression avec $n_1$ variables |
|---|---|
| $X_{train,k}$ | $(n_1,720)$ |
| $Y_{train,k}$ | $(1,720)$ |
| $X_{test,k}$ | $(n_1,180)$ |
| $Y_{test,k}$ | $(1,180)$ |

## Prédiction finale avec 5 modèles

Si on entraîne 5 modèles :

$$
Model_1,\ Model_2,\ Model_3,\ Model_4,\ Model_5
$$

La prédiction finale est la moyenne :

$$
\hat{y}
=
\frac{1}{5}
\sum_{i=1}^{5}
Model_i(x)
$$


In [ ]:
def make_5_folds_columns(X, Y):
    """
    Crée 5 folds pour des données stockées en colonnes.
    X : shape (n_features, 900)
    Y : shape (n_outputs, 900)
    """
    m = X.shape[1]
    assert m % 5 == 0, "Le nombre de données doit être divisible par 5."
    fold_size = m // 5

    folds = []
    for k in range(5):
        start = k * fold_size
        end = (k + 1) * fold_size

        X_test_k = X[:, start:end]
        Y_test_k = Y[:, start:end]

        X_train_k = np.concatenate([X[:, :start], X[:, end:]], axis=1)
        Y_train_k = np.concatenate([Y[:, :start], Y[:, end:]], axis=1)

        folds.append((X_train_k, Y_train_k, X_test_k, Y_test_k))

    return folds

def average_predictions(predictions):
    """
    predictions : liste de prédictions de plusieurs modèles.
    """
    return np.mean(np.array(predictions), axis=0)


# 13. Annale 2025 : régression sur les performances étudiantes

Le sujet demande de prédire une note à partir de 14 paramètres pour 1000 étudiants.

Donc :

$$
n_1 = 14
$$

$$
m = 1000
$$

La sortie est une note réelle, donc c’est une régression :

$$
n_3 = 1
$$

## Question 1 : dimensions

Si la couche cachée contient $n_2$ neurones :

| Objet | Dimension |
|---|---|
| $X$ | $(14,1000)$ |
| $Y$ | $(1,1000)$ |
| $a^{(2)}$ | $(n_2,1000)$ |
| $a^{(3)}$ | $(1,1000)$ |
| $W^{(2)}$ | $(n_2,14)$ |
| $b^{(2)}$ | $(n_2,1)$ |
| $W^{(3)}$ | $(1,n_2)$ |
| $b^{(3)}$ | $(1,1)$ |

Si $n_2 = 20$ :

| Objet | Dimension |
|---|---|
| $W^{(2)}$ | $(20,14)$ |
| $b^{(2)}$ | $(20,1)$ |
| $W^{(3)}$ | $(1,20)$ |
| $b^{(3)}$ | $(1,1)$ |

## Question 2 : séparation et normalisation

On sépare :

$$
X_{train} \in \mathbb{R}^{14 \times 900}
$$

$$
Y_{train} \in \mathbb{R}^{1 \times 900}
$$

$$
X_{test} \in \mathbb{R}^{14 \times 100}
$$

$$
Y_{test} \in \mathbb{R}^{1 \times 100}
$$

Puis on normalise les entrées entre 0 et 1 en utilisant les min et max du train.

## Question 3 : modèle de régression

Paramètres donnés :

$$
n_2 = 20
$$

$$
\alpha = 0.2
$$

$$
N_{iter} = 800
$$

À faire :

1. initialiser $W^{(2)}, b^{(2)}, W^{(3)}, b^{(3)}$ ;
2. faire la propagation avant ;
3. calculer la MSE ;
4. faire la backpropagation ;
5. mettre à jour les paramètres ;
6. tracer la fonction coût ;
7. calculer la MSE sur les 100 données test.

## Question 4 : validation croisée

On part des 900 données train et on fait 5 blocs.

Chaque bloc contient :

$$
180
$$

données de test interne.

Pour chaque modèle :

$$
X_{train,k} \in \mathbb{R}^{14 \times 720}
$$

$$
Y_{train,k} \in \mathbb{R}^{1 \times 720}
$$

$$
X_{test,k} \in \mathbb{R}^{14 \times 180}
$$

$$
Y_{test,k} \in \mathbb{R}^{1 \times 180}
$$

## Question 5 : modèle final

On entraîne 5 modèles puis on prédit avec :

$$
\hat{y}
=
\frac{1}{5}
\sum_{i=1}^{5}
Model_i(x)
$$

On évalue ensuite cette moyenne sur les 100 données test finales.


# 14. Annale 2024 : classification Iris

Le dataset Iris contient :

- 150 données ;
- 4 caractéristiques ;
- 3 classes.

Donc :

$$
n_1 = 4
$$

$$
K = 3
$$

Le modèle donné contient deux couches cachées puis une sortie softmax :

$$
a^{(1)} = X
$$

$$
z^{(2)} = W^{(2)}a^{(1)} + b^{(2)}
$$

$$
a^{(2)} = \sigma(z^{(2)})
$$

$$
z^{(3)} = W^{(3)}a^{(2)} + b^{(3)}
$$

$$
a^{(3)} = \sigma(z^{(3)})
$$

$$
z^{(4)} = W^{(4)}a^{(3)} + b^{(4)}
$$

$$
a^{(4)} = softmax(z^{(4)})
$$

## Dimensions

Si la couche 2 contient $n_2$ neurones et la couche 3 contient $n_3$ neurones :

| Objet | Dimension |
|---|---|
| $X$ | $(4,m)$ |
| $Y$ | $(3,m)$ |
| $W^{(2)}$ | $(n_2,4)$ |
| $b^{(2)}$ | $(n_2,1)$ |
| $W^{(3)}$ | $(n_3,n_2)$ |
| $b^{(3)}$ | $(n_3,1)$ |
| $W^{(4)}$ | $(3,n_3)$ |
| $b^{(4)}$ | $(3,1)$ |
| $a^{(4)}$ | $(3,m)$ |

## Coût

La fonction coût est :

$$
J =
-\frac{1}{m}
\sum_{i=1}^{m}
\sum_{k=1}^{3}
y_k^{(i)}
\log(a_k^{(4)}(x^{(i)}))
+
\frac{\lambda}{2m}
\sum_p
\sum_{i,j}
(W_{ij}^{(p)})^2
$$

Ici, $p$ parcourt les matrices de poids :

$$
p \in \{2,3,4\}
$$

Les biais ne sont pas régularisés.

## Question classique : valeurs de $k$, $x$ et $p$

- $k$ parcourt les classes : $k \in \{1,2,3\}$.
- $x$ parcourt les observations d’apprentissage.
- $p$ parcourt les couches de poids régularisées : $p \in \{2,3,4\}$.

## Choix des hyperparamètres

On teste plusieurs valeurs de :

$$
n_3,\quad \lambda,\quad N_{iter}
$$

On garde le modèle avec la meilleure accuracy sur le validation set.  
Ensuite seulement, on calcule l’accuracy sur le test set.


# 15. TP1 : régression linéaire

## Hypothèse

$$
h_\theta(x) = \theta^T x
$$

## Fonction coût

$$
J(\theta)
=
\frac{1}{2m}
\sum_{i=1}^{m}
(h_\theta(x^{(i)}) - y^{(i)})^2
$$

## Descente de gradient

$$
\theta
:=
\theta
-
\alpha
\frac{1}{m}
X^T(X\theta - y)
$$

## Normalisation

Si les variables ont des échelles différentes :

$$
X_{norm}
=
\frac{X-\mu}{\sigma}
$$


In [ ]:
def compute_cost_linear_regression(X, y, theta):
    m = y.size
    predictions = X @ theta
    return (1/(2*m)) * np.sum((predictions - y)**2)

def gradient_descent_linear_regression(X, y, theta, alpha, num_iters):
    m = y.size
    cost_history = []

    for _ in range(num_iters):
        theta = theta - alpha * (1/m) * X.T @ (X @ theta - y)
        cost_history.append(compute_cost_linear_regression(X, y, theta))

    return theta, cost_history


# 16. TP2 : régression logistique

## Hypothèse

$$
h_\theta(x)
=
\sigma(\theta^T x)
$$

avec :

$$
\sigma(z)
=
\frac{1}{1+e^{-z}}
$$

## Fonction coût

$$
J(\theta)
=
-\frac{1}{m}
\sum_{i=1}^{m}
\left[
y^{(i)}\log(h_\theta(x^{(i)}))
+
(1-y^{(i)})\log(1-h_\theta(x^{(i)}))
\right]
$$

## Gradient

$$
\nabla J(\theta)
=
\frac{1}{m}
X^T(h_\theta(X)-y)
$$

## Prédiction

$$
\hat{y}
=
\begin{cases}
1 & \text{si } h_\theta(x) \geq 0.5 \\
0 & \text{sinon}
\end{cases}
$$


In [ ]:
def logistic_cost(theta, X, y):
    m = y.size
    h = sigmoid(X @ theta)
    eps = 1e-12
    return -(1/m) * np.sum(y*np.log(h + eps) + (1-y)*np.log(1-h + eps))

def logistic_gradient(theta, X, y):
    m = y.size
    h = sigmoid(X @ theta)
    return (1/m) * X.T @ (h - y)

def logistic_predict(theta, X):
    return (sigmoid(X @ theta) >= 0.5).astype(int)


# 17. TP3 : one-vs-all

Le one-vs-all sert à faire de la classification multiclasses avec plusieurs régressions logistiques.

Si on a $K$ classes, on entraîne $K$ classifieurs :

$$
\theta^{(1)}, \theta^{(2)}, \ldots, \theta^{(K)}
$$

Chaque classifieur apprend :

$$
y = 1
$$

pour sa classe, et :

$$
y = 0
$$

pour toutes les autres.

## Prédiction

On choisit la classe avec la plus grande probabilité :

$$
\hat{y}
=
\arg\max_k
h_{\theta^{(k)}}(x)
$$


In [ ]:
def predict_one_vs_all(all_theta, X):
    """
    all_theta : shape (K, n+1)
    X : shape (m, n+1)
    """
    probabilities = sigmoid(X @ all_theta.T)
    return np.argmax(probabilities, axis=1)


# 18. TP6 : biais, variance et learning curves

## Underfitting

Le modèle est trop simple.

Signes :

- erreur train élevée ;
- erreur validation élevée ;
- les deux erreurs sont proches.

## Overfitting

Le modèle apprend trop les données d’entraînement.

Signes :

- erreur train faible ;
- erreur validation élevée ;
- gros écart entre train et validation.

## Bon modèle

Signes :

- erreur train faible ;
- erreur validation faible ;
- les deux erreurs sont proches.

## Rôle de $\lambda$

| Valeur de $\lambda$ | Effet |
|---|---|
| $\lambda = 0$ | pas de régularisation, risque d’overfitting |
| $\lambda$ modéré | meilleure généralisation |
| $\lambda$ trop grand | underfitting |


# 19. TP8 : K-means

K-means sert à regrouper des données en $K$ clusters.

## Algorithme

1. Initialiser $K$ centroïdes.
2. Assigner chaque point au centroïde le plus proche.
3. Recalculer chaque centroïde comme la moyenne des points assignés.
4. Répéter.

## Application à la compression d’image

Chaque pixel couleur est un point :

$$
(R,G,B)
$$

On choisit par exemple :

$$
K = 16
$$

Puis chaque pixel est remplacé par la couleur du centroïde le plus proche.

Résultat : l’image est représentée avec seulement 16 couleurs.


In [ ]:
def find_closest_centroids(X, centroids):
    """
    X : shape (m, n)
    centroids : shape (K, n)
    """
    distances = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
    return np.argmin(distances, axis=1)

def compute_centroids(X, idx, K):
    n = X.shape[1]
    centroids = np.zeros((K, n))

    for k in range(K):
        points = X[idx == k]
        if len(points) > 0:
            centroids[k] = np.mean(points, axis=0)

    return centroids


# 20. TP8 : PCA

PCA sert à réduire la dimension des données.

## Étapes

1. Normaliser les données.
2. Calculer la matrice de covariance.
3. Appliquer la SVD.
4. Garder les $K$ premières directions principales.
5. Projeter les données.
6. Reconstruire une approximation.

## Projection

Si $U_{reduce}$ contient les $K$ premières directions :

$$
Z = XU_{reduce}
$$

## Reconstruction

$$
X_{rec} = ZU_{reduce}^T
$$

## Interprétation

- Plus $K$ est petit, plus la compression est forte.
- Plus $K$ est grand, meilleure est la reconstruction.


In [ ]:
def feature_normalize(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    sigma[sigma == 0] = 1
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

def pca(X):
    m = X.shape[0]
    Sigma = (1/m) * X.T @ X
    U, S, Vt = np.linalg.svd(Sigma)
    return U, S

def project_data(X, U, K):
    U_reduce = U[:, :K]
    return X @ U_reduce

def recover_data(Z, U, K):
    U_reduce = U[:, :K]
    return Z @ U_reduce.T


# 21. Réponses types à connaître

## Pourquoi utiliser une sigmoïde en classification binaire ?

La sigmoïde renvoie une valeur entre 0 et 1.  
On peut donc interpréter la sortie comme une probabilité d’appartenir à la classe 1.

## Pourquoi utiliser softmax ?

Softmax transforme les sorties en probabilités dont la somme vaut 1.  
On peut donc choisir la classe avec la probabilité la plus grande.

## Pourquoi faire du one-hot encoding ?

Parce que la sortie d’un réseau softmax est un vecteur avec une composante par classe.  
Il faut donc que $Y$ ait la même dimension que la sortie.

## Pourquoi normaliser ?

La normalisation met les variables sur des échelles comparables.  
Cela aide l’apprentissage par descente de gradient.

## Pourquoi mélanger avant de séparer ?

Si les données sont rangées par classe ou par ordre, une séparation directe peut produire un train ou un test non représentatif.

## Pourquoi séparer train, validation et test ?

- Train : apprend les paramètres.
- Validation : choisit les hyperparamètres.
- Test : mesure la performance finale.

## Pourquoi ne pas utiliser le test set pour choisir les hyperparamètres ?

Parce que le test set doit rester indépendant.  
Sinon, il ne mesure plus correctement la généralisation.

## Comment reconnaître l’overfitting ?

Si la performance train est très bonne mais la performance validation/test est nettement moins bonne.

## Quel est l’effet de $\lambda$ ?

$\lambda$ contrôle la régularisation.  
S’il est trop faible, risque d’overfitting.  
S’il est trop fort, risque d’underfitting.


# 22. Checklist finale avant de répondre

Pour chaque question, vérifier :

- Ai-je identifié le type de problème ?
- Ai-je écrit les bonnes dimensions ?
- Ai-je choisi la bonne dernière activation ?
- Ai-je choisi la bonne fonction coût ?
- Ai-je utilisé la bonne métrique ?
- Ai-je ajouté la régularisation seulement si $\lambda$ existe ?
- Ai-je régularisé seulement les poids et pas les biais ?
- Ai-je utilisé validation seulement pour choisir les hyperparamètres ?
- Ai-je gardé le test set pour la fin ?
- Ai-je vérifié que tous les produits matriciels sont compatibles ?
